# Phase 6: Feature Engineering with HOG and PCA (Q38–Q42)

**Questions covered:** Q38 (grayscale vs RGB), Q39 (HOG features), Q40 (HOG + SVM vs raw pixels), Q41 (PCA, variance retained), Q42 (PCA + classifier).

**Rules carried over from Phase 5**
- Train on the **train** split, tune on **validation**. The **test** split is never touched here. (HOG is computed for every image because it is a fixed per-image function, but no test row is ever indexed or scored.)
- PCA and every scaler are **fitted on training data only**.
- Models run on the GPU where the algorithm allows it (Logistic Regression, KNN, approximate RBF SVM). Random Forest stays on CPU. The SVM is a **random-Fourier-feature approximation of an RBF SVM**, not exact libsvm. Say so in the report.
- Hyperparameters are tuned on validation accuracy, so validation scores are slightly optimistic. The test set stays untouched until Phase 9.

**Needs from earlier phases:** `data/processed/images_64.npy`, `images_128.npy`, `labels.npy`, `data/splits/split_seed42.csv`, `results/metrics/05_traditional_configs.json`, and `src/evaluation.py`, `traditional_models.py`, `gpu_models.py`.

In [ ]:
import os
for var in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"]:
    os.environ[var] = "8"          # the container is limited to 8 CPUs
import sys, json, time, random, importlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from pathlib import Path

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.set_num_threads(8)

PROJECT_ROOT = Path("/workspace/sic_final_project")      # change here if the folder moves
sys.path.insert(0, str(PROJECT_ROOT / "src"))
MODEL_DIR = PROJECT_ROOT / "models/traditional"
for d in [MODEL_DIR, PROJECT_ROOT / "results/figures", PROJECT_ROOT / "results/metrics", PROJECT_ROOT / "results/comparisons"]:
    d.mkdir(parents=True, exist_ok=True)

needed = ["data/processed/images_64.npy", "data/processed/images_128.npy", "data/processed/labels.npy",
          "data/splits/split_seed42.csv", "results/metrics/05_traditional_configs.json",
          "src/evaluation.py", "src/traditional_models.py", "src/gpu_models.py"]
missing = [f for f in needed if not (PROJECT_ROOT / f).exists()]
for f in needed:
    print("OK     " if f not in missing else "MISSING", f)
assert not missing, "Run Phases 4 and 5 first (see MISSING above)."

## Step 1: write the reusable modules

`feature_extraction.py` holds grayscale conversion, HOG extraction (parallel on CPU) and a GPU PCA.
`experiments.py` holds a small runner that fits, evaluates, times and logs models, so the notebook stays short.

In [ ]:
%%writefile /workspace/sic_final_project/src/feature_extraction.py
"""Feature extraction: grayscale, HOG (CPU, multiprocess) and PCA (GPU)."""
import multiprocessing as mp
import numpy as np
import torch
from skimage.feature import hog

GRAY_WEIGHTS = np.array([0.299, 0.587, 0.114], dtype=np.float32)    # ITU-R 601 luma
HOG_PARAMS = dict(orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2),
                  block_norm="L2-Hys", feature_vector=True)


def to_gray(images_u8, batch=2048):
    """(N, H, W, 3) uint8 -> (N, H, W) uint8 using luma weights. Works on memory-mapped arrays."""
    out = np.empty(images_u8.shape[:3], dtype=np.uint8)
    for s in range(0, len(images_u8), batch):
        b = np.asarray(images_u8[s:s + batch], dtype=np.float32)
        out[s:s + batch] = np.rint(b @ GRAY_WEIGHTS).astype(np.uint8)
    return out


_SHARED = None          # set before the pool forks, so children read the images without copying


def _hog_range(bounds):
    s, e = bounds
    return np.stack([hog(_SHARED[i], **HOG_PARAMS) for i in range(s, e)]).astype(np.float32)


def hog_features(gray_u8, n_jobs=8, chunk=256):
    """HOG descriptor for every image of a (N, H, W) uint8 array. Uses fork + pipes (no /dev/shm needed)."""
    global _SHARED
    _SHARED = gray_u8
    bounds = [(s, min(s + chunk, len(gray_u8))) for s in range(0, len(gray_u8), chunk)]
    with mp.get_context("fork").Pool(n_jobs) as pool:
        parts = pool.map(_hog_range, bounds)
    _SHARED = None
    return np.concatenate(parts)


def hog_visual(gray_img):
    """Return (descriptor, visualisation image) for one grayscale image."""
    return hog(gray_img, visualize=True, **HOG_PARAMS)


class GPUPCA:
    """PCA via randomised truncated SVD (torch.svd_lowrank) on the centred data.
    Fit on TRAINING data only. explained_variance_ratio_ uses the exact total variance (trace of the covariance),
    so cumulative ratios are comparable with scikit-learn's PCA(svd_solver='randomized')."""

    def __init__(self, n_components=1000, niter=6, seed=42):
        self.n_components_, self.niter, self.seed = n_components, niter, seed

    def fit(self, X):
        n = X.shape[0]
        self.mean = X.mean(dim=0, keepdim=True)
        Xc = X - self.mean
        total_var = float(torch.linalg.vector_norm(Xc.flatten(), dtype=torch.float64) ** 2) / (n - 1)
        torch.manual_seed(self.seed)
        _, S, V = torch.svd_lowrank(Xc, q=self.n_components_, niter=self.niter)
        del Xc
        self.components = V.T.contiguous()                        # (k, d)
        self.explained_variance = S.double() ** 2 / (n - 1)
        self.explained_variance_ratio = (self.explained_variance / total_var).float()
        self.total_var = total_var
        return self

    def transform(self, X, k=None):
        k = k or self.components.shape[0]
        return (X - self.mean) @ self.components[:k].T

    def inverse_transform(self, Z):
        k = Z.shape[1]
        return Z @ self.components[:k] + self.mean

    def n_components_for(self, target):
        cum = torch.cumsum(self.explained_variance_ratio, dim=0)
        hit = (cum >= target).nonzero()
        return int(hit[0]) + 1 if len(hit) else None

In [ ]:
%%writefile /workspace/sic_final_project/src/experiments.py
"""Small helper that fits, times, evaluates and logs tabular-feature models (GPU or CPU)."""
import time
import numpy as np
import pandas as pd
import torch
import evaluation
import gpu_models


class TabularRunner:
    def __init__(self, X_train, y_train, X_val, y_val, device=None, seed=42, n_train_acc=3000, store=None):
        self.dev = device or gpu_models.DEVICE
        self.X_train, self.y_train, self.X_val, self.y_val = X_train, y_train, X_val, y_val
        self.Xg_train = torch.from_numpy(np.ascontiguousarray(X_train, dtype=np.float32)).to(self.dev)
        self.yg_train = torch.from_numpy(y_train).to(self.dev)
        self.Xg_val = torch.from_numpy(np.ascontiguousarray(X_val, dtype=np.float32)).to(self.dev)
        self.sub = np.random.default_rng(seed).choice(len(X_train), min(n_train_acc, len(X_train)), replace=False)
        self.sub_t = torch.from_numpy(self.sub).to(self.dev)
        self.preds = store if store is not None else {}      # name -> (y_pred, score)
        self.models = {}
        _ = self.Xg_val[:64] @ self.Xg_val[:64].T            # warm-up, so CUDA start-up is not timed
        self._sync()

    def _sync(self):
        if self.dev.type == "cuda":
            torch.cuda.synchronize()

    def val_acc(self, model):
        s = model.score_dog(self.Xg_val)
        return float(((s > model.threshold).astype(int) == self.y_val).mean())

    def tune(self, make, values, label):
        """Fit make(v) for each v; return (table, best value by validation accuracy)."""
        rows = []
        for v in values:
            self._sync(); t0 = time.perf_counter()
            m = make(v).fit(self.Xg_train, self.yg_train); self._sync()
            rows.append({label: v, "val_acc": self.val_acc(m), "fit_s": round(time.perf_counter() - t0, 1)})
            del m
        table = pd.DataFrame(rows)
        return table, table.loc[table["val_acc"].idxmax(), label]

    def tune_knn(self, ks):
        """Neighbours are searched once and reused for every k (odd k only, so there are no ties)."""
        knn = gpu_models.GPUKNN().fit(self.Xg_train, self.yg_train)
        idx = knn.neighbours(self.Xg_val, max(ks))
        votes = self.yg_train[idx].float()
        rows = [{"k": k, "val_acc": float(((votes[:, :k].mean(1) > 0.5).cpu().numpy().astype(int) == self.y_val).mean())}
                for k in ks]
        table = pd.DataFrame(rows)
        return table, int(table.loc[table["val_acc"].idxmax(), "k"])

    def run_gpu(self, name, model, notes="", train_acc=True):
        self._sync(); t0 = time.perf_counter()
        model.fit(self.Xg_train, self.yg_train); self._sync()
        fit_s = time.perf_counter() - t0
        t0 = time.perf_counter(); score = model.score_dog(self.Xg_val); self._sync()
        ms = 1000 * (time.perf_counter() - t0) / len(self.Xg_val)
        y_pred = (score > model.threshold).astype(int)
        metrics = evaluation.compute_metrics(self.y_val, y_pred, score)
        tr_acc = np.nan
        if train_acc:
            s_tr = model.score_dog(self.Xg_train[self.sub_t])
            tr_acc = float(((s_tr > model.threshold).astype(int) == self.y_train[self.sub]).mean())
        evaluation.log_result(name, "val", metrics, train_accuracy=tr_acc, train_time_s=round(fit_s, 1),
                              infer_ms_per_img=round(ms, 4), n_features=int(self.Xg_train.shape[1]),
                              notes=notes + " | GPU (PyTorch)")
        self.preds[name], self.models[name] = (y_pred, score), model
        print(f"{name}: val acc {metrics['accuracy']:.4f} | train acc (3k) {tr_acc:.4f} | AUC {metrics['roc_auc']:.4f} | "
              f"fit {fit_s:.1f}s | {ms:.4f} ms/img | {self.Xg_train.shape[1]} features")
        return metrics

    def run_cpu(self, name, model, notes=""):
        t0 = time.perf_counter(); model.fit(self.X_train, self.y_train); fit_s = time.perf_counter() - t0
        metrics, y_pred, score, ms = evaluation.evaluate_sklearn(model, self.X_val, self.y_val)
        tr_acc = float(model.score(self.X_train[self.sub], self.y_train[self.sub]))
        evaluation.log_result(name, "val", metrics, train_accuracy=tr_acc, train_time_s=round(fit_s, 1),
                              infer_ms_per_img=round(ms, 4), n_features=int(self.X_train.shape[1]),
                              notes=notes + " | CPU (scikit-learn)")
        self.preds[name], self.models[name] = (y_pred, score), model
        print(f"{name}: val acc {metrics['accuracy']:.4f} | train acc (3k) {tr_acc:.4f} | AUC {metrics['roc_auc']:.4f} | "
              f"fit {fit_s:.1f}s | {ms:.4f} ms/img | {self.X_train.shape[1]} features")
        return metrics

    def close(self):
        for a in ("Xg_train", "yg_train", "Xg_val", "sub_t"):
            if hasattr(self, a):
                delattr(self, a)
        self.models.clear()
        if self.dev.type == "cuda":
            torch.cuda.empty_cache()

## Step 2: load the data and the Phase 5 settings

In [ ]:
import evaluation, traditional_models, gpu_models, feature_extraction, experiments
for m in (evaluation, traditional_models, gpu_models, feature_extraction, experiments):
    importlib.reload(m)
dev = gpu_models.DEVICE
print("device:", dev, "|", torch.cuda.get_device_name(0) if dev.type == "cuda" else "CPU ONLY (tell your mentor, the GPU is not visible)")

cfg5 = json.loads((PROJECT_ROOT / "results/metrics/05_traditional_configs.json").read_text())
print("Phase 5 settings:", json.dumps(cfg5))

split_df = pd.read_csv(PROJECT_ROOT / "data/splits/split_seed42.csv")
labels = np.load(PROJECT_ROOT / "data/processed/labels.npy")
tr = split_df.loc[split_df["split"] == "train", "array_index"].to_numpy()
va = split_df.loc[split_df["split"] == "val", "array_index"].to_numpy()
y_train, y_val = labels[tr], labels[va]
print("train", len(tr), "| val", len(va), "| (test rows are never indexed in this notebook)")

imgs64 = np.load(PROJECT_ROOT / "data/processed/images_64.npy", mmap_mode="r")
gray64 = feature_extraction.to_gray(imgs64)               # (N, 64, 64) uint8; a per-image operation
print("gray64:", gray64.shape, gray64.dtype)
VAL_PREDS = {}                                              # name -> (y_pred, score), for confusion matrices later

## Q38: Grayscale vs RGB

Grayscale keeps brightness and edges but drops colour, and it cuts the features from 64×64×3 = **12,288** to 64×64 = **4,096**.
Question: does colour carry useful information for cats vs dogs, or is it mostly noise (lighting, backgrounds)?

Fairness note: each model reuses the hyperparameters that were tuned on the **RGB** features in Phase 5 (no retuning), so RGB has a small advantage. The comparison is still informative.

In [ ]:
Xg_tr = traditional_models.flatten(gray64[tr])
Xg_va = traditional_models.flatten(gray64[va])
print("Q38 - grayscale features per image:", Xg_tr.shape[1], "| RGB:", 64 * 64 * 3)

runner = experiments.TabularRunner(Xg_tr, y_train, Xg_va, y_val, dev, store=VAL_PREDS)
runner.run_gpu("LR - Grayscale", gpu_models.GPULogisticRegression(cfg5["logreg"]["C"]),
               notes=f"grayscale 64x64, C={cfg5['logreg']['C']} (reused from RGB)")
runner.run_gpu("KNN - Grayscale", gpu_models.GPUKNN(k=cfg5["knn"]["k"]),
               notes=f"grayscale 64x64, k={cfg5['knn']['k']} (reused from RGB)", train_acc=False)
runner.run_gpu("SVM - Grayscale", gpu_models.GPURBFSVM(C=cfg5["svm"]["C"], n_components=cfg5["svm"]["n_components"]),
               notes=f"APPROX RBF SVM on grayscale 64x64, C={cfg5['svm']['C']} (reused from RGB)")
RUN_RF_GRAY = True        # Random Forest runs on CPU: set False to skip if it is too slow
if RUN_RF_GRAY:
    runner.run_cpu("RF - Grayscale", traditional_models.make_rf(cfg5["rf"]["n_estimators"]),
                   notes=f"grayscale 64x64, {cfg5['rf']['n_estimators']} trees")
runner.close(); del Xg_tr, Xg_va

In [ ]:
table = pd.read_csv(evaluation.RESULTS_CSV)
val = table[table["split"] == "val"].set_index("experiment")
pairs = [("Logistic Regression", "LR - Grayscale"), ("KNN", "KNN - Grayscale"),
         ("SVM - Raw Pixels", "SVM - Grayscale"), ("Random Forest", "RF - Grayscale")]
rows = []
for rgb, gray in pairs:
    if rgb in val.index and gray in val.index:
        rows.append({"model": rgb.replace(" - Raw Pixels", ""),
                     "RGB val acc": val.loc[rgb, "accuracy"], "Gray val acc": val.loc[gray, "accuracy"],
                     "diff (pp)": 100 * (val.loc[gray, "accuracy"] - val.loc[rgb, "accuracy"]),
                     "RGB AUC": val.loc[rgb, "roc_auc"], "Gray AUC": val.loc[gray, "roc_auc"],
                     "RGB fit s": val.loc[rgb, "train_time_s"], "Gray fit s": val.loc[gray, "train_time_s"]})
q38 = pd.DataFrame(rows)
display(q38.round(4))

if len(q38):
    ax = q38.set_index("model")[["RGB val acc", "Gray val acc"]].plot.bar(figsize=(7, 4), rot=0)
    ax.set_ylim(0.4, max(0.7, q38[["RGB val acc", "Gray val acc"]].values.max() + 0.05))
    ax.axhline(0.5, color="gray", ls="--", lw=1); ax.set_title("Q38: RGB vs grayscale (validation accuracy)")
    plt.tight_layout(); plt.savefig(PROJECT_ROOT / "results/figures/06_gray_vs_rgb.png", dpi=110); plt.show()

## Q39: HOG features

**Histogram of Oriented Gradients** describes *local edge directions* instead of raw brightness:
1. Compute the brightness gradient (direction and strength) at each pixel.
2. Split the image into small **cells** (here 8×8 pixels) and, per cell, build a histogram of gradient directions (**9 orientation bins**), weighted by gradient strength.
3. Group cells into overlapping **blocks** (2×2 cells) and normalise each block (L2-Hys), which makes the descriptor robust to local lighting changes.
4. Concatenate all block histograms into one vector.

We use **128×128 grayscale** images: 16×16 cells → 15×15 blocks × (2×2 cells × 9 bins) = **8,100 features** per image.
HOG is a *hand-designed* feature extractor, so a classical classifier gets shape information that raw pixels hide.

In [ ]:
imgs128 = np.load(PROJECT_ROOT / "data/processed/images_128.npy", mmap_mode="r")
gray128 = feature_extraction.to_gray(imgs128)            # (N, 128, 128) uint8
hog_path = PROJECT_ROOT / "data/processed/hog_gray128.npy"

if hog_path.exists():
    hog_all = np.load(hog_path)
    print("loaded cached HOG features", hog_all.shape)
else:
    t0 = time.perf_counter(); _ = feature_extraction.hog_features(gray128[:256], n_jobs=8)
    t_probe = time.perf_counter() - t0
    print(f"probe: 256 images in {t_probe:.1f}s -> rough estimate for all {len(gray128)}: {t_probe * len(gray128) / 256 / 60:.1f} min")
    t0 = time.perf_counter()
    hog_all = feature_extraction.hog_features(gray128, n_jobs=8)
    print(f"HOG for {len(hog_all)} images in {time.perf_counter() - t0:.0f}s -> {hog_all.shape}, {hog_all.nbytes / 1e9:.2f} GB")
    np.save(hog_path, hog_all)
print("HOG features per image:", hog_all.shape[1], "| expected 15*15*2*2*9 =", 15 * 15 * 2 * 2 * 9)
assert hog_all.shape[0] == len(split_df) and np.isfinite(hog_all).all()

In [ ]:
# What does HOG "see"? (two cats, two dogs from the validation split)
pick = [int(va[np.where(y_val == c)[0][j]]) for c in (0, 1) for j in (3, 11)]
fig, ax = plt.subplots(2, 4, figsize=(13, 6.5))
for j, i in enumerate(pick):
    _, himg = feature_extraction.hog_visual(gray128[i])
    ax[0, j].imshow(gray128[i], cmap="gray"); ax[0, j].set_title(["Cat", "Dog"][labels[i]])
    ax[1, j].imshow(himg, cmap="gray"); ax[1, j].set_title("HOG cells (edge directions)")
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig(PROJECT_ROOT / "results/figures/06_hog_visual.png", dpi=110); plt.show()

## Q40: HOG + SVM

The SVM is the approximate RBF SVM from Phase 5. We tune only `C` (validation accuracy), with 4,096 random Fourier features, so this is a like-for-like comparison with `SVM - Raw Pixels`.

In [ ]:
runner = experiments.TabularRunner(hog_all[tr], y_train, hog_all[va], y_val, dev, store=VAL_PREDS)
hog_tab, best_hog_C = runner.tune(lambda C: gpu_models.GPURBFSVM(C=C, n_components=4096), [0.1, 1.0, 10.0], "C")
display(hog_tab)
hog_metrics = runner.run_gpu("HOG + SVM", gpu_models.GPURBFSVM(C=best_hog_C, n_components=4096),
                             notes=f"HOG(128x128 gray, 9 orient, 8x8 cells, 2x2 blocks) + APPROX RBF SVM, C={best_hog_C}, D=4096")
runner.close()

In [ ]:
table = pd.read_csv(evaluation.RESULTS_CSV)
val = table[table["split"] == "val"].set_index("experiment")
cmp40 = val.loc[[n for n in ["SVM - Raw Pixels", "SVM - Grayscale", "HOG + SVM"] if n in val.index],
                ["accuracy", "precision", "recall", "f1", "roc_auc", "train_accuracy", "train_time_s", "n_features"]]
display(cmp40.round(4))

from sklearn.metrics import classification_report
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, n in zip(axes, [n for n in ["SVM - Raw Pixels", "HOG + SVM"] if n in VAL_PREDS]):
    cm = evaluation.plot_confusion(y_val, VAL_PREDS[n][0], n, ax=ax)
plt.tight_layout(); plt.savefig(PROJECT_ROOT / "results/figures/06_hog_confusion.png", dpi=110); plt.show()
if "HOG + SVM" in VAL_PREDS:
    print(classification_report(y_val, VAL_PREDS["HOG + SVM"][0], target_names=evaluation.CLASS_NAMES, digits=4))

## Q41: PCA and variance retained

**PCA** finds new orthogonal axes (principal components) ordered by how much of the data's variance they capture. Keeping the first *k* components gives a compressed *k*-number description of every image.
The PCA is **fitted on the training images only**; validation images are projected with the same axes.

Implementation note: the SVD is a *randomised* truncated SVD of the centred data (top 1,000 components), run on the GPU. The total variance is computed exactly, so the cumulative-variance numbers are meaningful as long as 1,000 components reach the targets (the code checks this).

In [ ]:
imgs_flat_train = traditional_models.flatten(imgs64[tr])      # (n_train, 12288) RGB 64x64, scaled 0-1
Xg = torch.from_numpy(imgs_flat_train).to(dev)
t0 = time.perf_counter()
pca = feature_extraction.GPUPCA(n_components=1000).fit(Xg)
if dev.type == "cuda": torch.cuda.synchronize()
print(f"PCA fitted on {Xg.shape[0]} training images in {time.perf_counter() - t0:.1f}s")

evr = pca.explained_variance_ratio.cpu().numpy()
cum = np.cumsum(evr)
k90, k95 = pca.n_components_for(0.90), pca.n_components_for(0.95)
print("variance captured by the first component:", round(float(evr[0]), 4), "| first 10:", round(float(cum[9]), 4),
      "| all 1000:", round(float(cum[-1]), 4))
if k95 is None or k90 is None:
    print("WARNING: 1000 components do not reach the target variance. Raise n_components before trusting k90/k95.")
k90 = k90 or 1000; k95 = k95 or 1000
D = Xg.shape[1]
print(f"\nQ41 - components for 90% variance: {k90}  ({100 * (1 - k90 / D):.1f}% fewer features than {D})")
print(f"Q41 - components for 95% variance: {k95}  ({100 * (1 - k95 / D):.1f}% fewer features than {D})")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(np.arange(1, 1001), cum); ax[0].axhline(0.90, ls="--", c="gray"); ax[0].axhline(0.95, ls="--", c="gray")
ax[0].axvline(k90, ls=":", c="C1"); ax[0].axvline(k95, ls=":", c="C2")
ax[0].set(xlabel="number of components", ylabel="cumulative explained variance", title=f"90% at {k90}, 95% at {k95}")
ax[1].bar(range(1, 31), evr[:30]); ax[1].set(xlabel="component", ylabel="explained variance ratio", title="First 30 components")
plt.tight_layout(); plt.savefig(PROJECT_ROOT / "results/figures/06_pca_variance.png", dpi=110); plt.show()

In [ ]:
# What do the components look like, and what is lost by compressing? (validation images)
comps = pca.components[:8].cpu().numpy().reshape(8, 64, 64, 3)
fig, ax = plt.subplots(1, 8, figsize=(16, 2.4))
for a, c in zip(ax, comps):
    a.imshow((c - c.min()) / (c.max() - c.min())); a.axis("off")
fig.suptitle("First 8 principal components (each is a 64x64x3 'pattern' that images are built from)")
plt.tight_layout(); plt.show()

Xv = torch.from_numpy(traditional_models.flatten(imgs64[va[:4]])).to(dev)
ks = [20, k90, k95]
fig, ax = plt.subplots(len(ks) + 1, 4, figsize=(9, 9))
for j in range(4):
    ax[0, j].imshow(Xv[j].cpu().numpy().reshape(64, 64, 3)); ax[0, j].set_title("original" if j == 0 else "")
    for r, k in enumerate(ks, start=1):
        rec = pca.inverse_transform(pca.transform(Xv[j:j + 1], k)).clamp(0, 1).cpu().numpy().reshape(64, 64, 3)
        ax[r, j].imshow(rec); ax[r, j].set_title(f"{k} components" if j == 0 else "")
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig(PROJECT_ROOT / "results/figures/06_pca_reconstruction.png", dpi=110); plt.show()

torch.save({"mean": pca.mean.cpu(), "components": pca.components[:k95].cpu(), "k90": int(k90), "k95": int(k95),
            "explained_variance_ratio": pca.explained_variance_ratio.cpu()}, MODEL_DIR / "pca_rgb64.pt")
print("saved PCA ->", MODEL_DIR / "pca_rgb64.pt", f"({(MODEL_DIR / 'pca_rgb64.pt').stat().st_size / 1e6:.0f} MB)")

## Q42: PCA + classifier

Transform the train and validation images to `k95` principal components (fit on train only), then train classifiers on the compressed vectors.

- **KNN** uses the PCA vectors directly (distances are preserved, so this is nearly the same neighbour search as on raw pixels, but much cheaper).
- **Logistic Regression and SVM** standardise their inputs inside `fit`. On PCA outputs, standardising each component to unit variance is effectively *whitening*, which boosts the weak components. Mention this in your discussion.

In [ ]:
Xv_all = torch.from_numpy(traditional_models.flatten(imgs64[va])).to(dev)
Ztr = pca.transform(Xg, k95).cpu().numpy()
Zva = pca.transform(Xv_all, k95).cpu().numpy()
del Xg, Xv_all, Xv; torch.cuda.empty_cache()
print("PCA features:", Ztr.shape, Zva.shape)

runner = experiments.TabularRunner(Ztr, y_train, Zva, y_val, dev, store=VAL_PREDS)
pca_res = {}

m = runner.run_gpu("PCA + KNN", gpu_models.GPUKNN(k=cfg5["knn"]["k"]),
                   notes=f"{k95} PCs (95% variance), k={cfg5['knn']['k']}, no whitening", train_acc=False)
pca_res["PCA + KNN"] = {"acc": m["accuracy"], "type": "knn", "k": int(cfg5["knn"]["k"])}

lr_tab, best_lr_C = runner.tune(lambda C: gpu_models.GPULogisticRegression(C), [1e-4, 1e-3, 1e-2, 1e-1], "C")
display(lr_tab)
m = runner.run_gpu("PCA + LR", gpu_models.GPULogisticRegression(best_lr_C), notes=f"{k95} PCs (95% variance), C={best_lr_C}")
pca_res["PCA + LR"] = {"acc": m["accuracy"], "type": "lr", "C": float(best_lr_C)}

svm_tab, best_svm_C = runner.tune(lambda C: gpu_models.GPURBFSVM(C=C, n_components=4096), [0.1, 1.0, 10.0], "C")
display(svm_tab)
m = runner.run_gpu("PCA + SVM", gpu_models.GPURBFSVM(C=best_svm_C, n_components=4096),
                   notes=f"{k95} PCs (95% variance) + APPROX RBF SVM, C={best_svm_C}, D=4096")
pca_res["PCA + SVM"] = {"acc": m["accuracy"], "type": "svm", "C": float(best_svm_C), "D": 4096}

best_pca_name = max(pca_res, key=lambda n: pca_res[n]["acc"])
print("\nBest PCA classifier on validation:", best_pca_name, round(pca_res[best_pca_name]["acc"], 4))

In [ ]:
# 90% vs 95% variance for the SVM: the 90% projection is simply the first k90 columns of the 95% projection
rows = []
for k in (k90, k95):
    runner._sync(); t0 = time.perf_counter()
    mdl = gpu_models.GPURBFSVM(C=best_svm_C, n_components=4096).fit(runner.Xg_train[:, :k].contiguous(), runner.yg_train)
    runner._sync(); fit_s = time.perf_counter() - t0
    s = mdl.score_dog(runner.Xg_val[:, :k].contiguous())
    rows.append({"variance kept": "90%" if k == k90 else "95%", "components": k,
                 "val_acc": float(((s > 0).astype(int) == y_val).mean()), "fit_s": round(fit_s, 1)})
    del mdl
display(pd.DataFrame(rows))
runner.close()

## Summary of Phase 6 and Phase 5 (validation set only)

In [ ]:
table = pd.read_csv(evaluation.RESULTS_CSV)
val = table[table["split"] == "val"].set_index("experiment")
names = ["Majority baseline", "Logistic Regression", "KNN", "SVM - Raw Pixels", "Random Forest",
         "LR - Grayscale", "KNN - Grayscale", "SVM - Grayscale", "RF - Grayscale",
         "HOG + SVM", "PCA + KNN", "PCA + LR", "PCA + SVM"]
summary = val.loc[[n for n in names if n in val.index],
                  ["accuracy", "precision", "recall", "f1", "roc_auc", "train_accuracy", "train_time_s", "infer_ms_per_img"]]
summary["train-val gap"] = summary["train_accuracy"] - summary["accuracy"]
display(summary.round(4))

for n in ["HOG + SVM", best_pca_name]:
    if n in VAL_PREDS:
        cm = evaluation.plot_confusion(y_val, VAL_PREDS[n][0], n)
        print(f"{n}: errors {cm[0, 1] + cm[1, 0]} (Cat->Dog {cm[0, 1]}, Dog->Cat {cm[1, 0]})")
        plt.show()

feature_cfg = {
    "gray": {"image_size": 64, "reused_from_rgb": True},
    "hog": {"gray_size": 128, "params": {"orientations": 9, "pixels_per_cell": [8, 8], "cells_per_block": [2, 2],
                                         "block_norm": "L2-Hys"},
            "n_features": int(hog_all.shape[1]), "svm": {"C": float(best_hog_C), "n_components": 4096}},
    "pca": {"fit_on": "train, RGB 64x64 flattened /255", "k90": int(k90), "k95": int(k95), "k_used": int(k95),
            "best_classifier": best_pca_name, "classifier": pca_res[best_pca_name], "all_classifiers": pca_res},
}
(PROJECT_ROOT / "results/metrics/06_feature_configs.json").write_text(json.dumps(feature_cfg, indent=2))
print("saved results/metrics/06_feature_configs.json")

## Your write-ups (use your own numbers and words)

**Q38.** State the RGB vs grayscale difference per model (use the `diff (pp)` column), and say what it suggests about colour: informative, or mostly noise/background? Mention the 3× reduction in features and the fit-time effect. Remember RGB models had tuned hyperparameters and grayscale reused them.

**Q39.** Explain HOG in your own words (gradients → cell histograms → block normalisation → one vector). Give the feature count (8,100) and why it suits shape (ears, snout) better than raw pixels.

**Q40.** Compare `HOG + SVM` with `SVM - Raw Pixels`: accuracy, F1, AUC, features, fit time, train-vs-val gap. Say whether hand-crafted features helped and by how much, and whether it is a large or small improvement.

**Q41.** Report `k90` and `k95` and the % reduction in dimensionality. Describe the shape of the cumulative-variance curve (how steep at the start?) and what the reconstruction images show about what is lost at 20 / k90 / k95 components.

**Q42.** Compare `PCA + <classifier>` against the same classifier on raw pixels (accuracy, fit time, inference time). Discuss whether compression hurt, helped or was neutral, and mention the whitening effect for LR/SVM.

**For the report:** the SVM is an *approximate* RBF SVM on the GPU; PCA is a randomised truncated SVD; hyperparameters were tuned on validation data; the test set is untouched.

Files created: `src/feature_extraction.py`, `src/experiments.py`, `data/processed/hog_gray128.npy`, `models/traditional/pca_rgb64.pt`, `results/metrics/06_feature_configs.json`, figures `06_*.png`, and new rows in `results/comparisons/model_results.csv`.